# 03. Fine-tuned transformer (DistilBERT)

Same features, splits and metrics as the LightGBM model carried forward from `02_baselines.ipynb` (validation PR-AUC 0.3412). The question: does a pretrained transformer, fine-tuned on transactions written as text, rank laundering better than gradient boosting on the same information?

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from bankguard.data import load_accounts, load_transactions, split_by_date
from bankguard.features import fit_currency_median
from bankguard.train import FEATURES, build_features

DATA = Path("../data")
print("GPU:", torch.cuda.get_device_name(0))

tx = load_transactions(DATA / "HI-Small_Trans.csv")
accounts = load_accounts(DATA / "HI-Small_accounts.csv")
splits = split_by_date(tx)
train, val = splits["train"], splits["validation"]

currency_median = fit_currency_median(train)
X = build_features(tx, accounts, currency_median)

X_train, y_train = X.loc[train.index, FEATURES], train["is_laundering"].to_numpy()
X_val, y_val = X.loc[val.index, FEATURES], val["is_laundering"].to_numpy()
print(X_train.shape, y_train.sum(), X_val.shape, y_val.sum())

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 3060 Laptop GPU
(3248921, 22) 2530 (965524, 22) 1036


Features are built with the `bankguard` package, so the transformer sees exactly the 22 features of the LightGBM model carried forward, computed by the same tested code. Any difference in score therefore comes from the learner, not from the inputs. The test split is not assigned to a variable and stays untouched until the final comparison.

Train has 3,248,921 rows with 2,530 laundering, validation 965,524 rows with 1,036, matching `02_baselines.ipynb`.

In [2]:
def _int(s, missing="unknown"):
    """Whole numbers as text, with missing values spelled out."""
    out = s.round().astype("Int64").astype(str)
    return out.where(s.notna(), missing)


def to_text(X: pd.DataFrame) -> pd.Series:
    """Write each row of the 22 features as one short sentence."""
    hour = (np.round(np.arctan2(X["hour_sin"], X["hour_cos"]) * 12 / np.pi) % 24).astype(int)
    yes_no = lambda s: np.where(s == 1, "yes", "no")
    return (
        "amount " + X["log_amount"].round(1).astype(str)
        + " log10, " + X["amount_vs_currency"].round(1).astype(str) + " vs typical for currency. "
        + "hour " + hour.astype(str) + ". "
        + np.where(X["self_transfer"] == 1, "self transfer. ", "")
        + "sender: " + _int(X["sender_out_1d"]) + " out, " + _int(X["sender_in_1d"]) + " in last day; "
        + _int(X["sender_out_3d"]) + " out, " + _int(X["sender_in_3d"]) + " in last 3 days; "
        + _int(X["sender_new_pair_out_3d"]) + " new payees. "
        + "receiver: " + _int(X["receiver_in_1d"]) + " in, " + _int(X["receiver_out_1d"]) + " out last day; "
        + _int(X["receiver_in_3d"]) + " in, " + _int(X["receiver_out_3d"]) + " out last 3 days; "
        + _int(X["receiver_new_pair_in_3d"]) + " new payers. "
        + "pair seen " + _int(X["pair_prior_3d"]) + " times. "
        + "sender " + X["sender_entity_type"].fillna("unknown").astype(str)
        + " with " + _int(X["sender_entity_accounts"]) + " accounts, "
        + "receiver " + X["receiver_entity_type"].fillna("unknown").astype(str)
        + " with " + _int(X["receiver_entity_accounts"]) + " accounts. "
        + "same owner " + yes_no(X["same_entity"]) + ", same bank " + yes_no(X["same_bank"]) + "."
    )


examples = X_train.loc[[train.index[y_train == 0][0], train.index[y_train == 1][0]]]
for label, text in zip(["normal", "laundering"], to_text(examples)):
    print(f"{label}:\n{text}\n")

normal:
amount 3.6 log10, 0.6 vs typical for currency. hour 0. self transfer. sender: 0 out, 0 in last day; 0 out, 0 in last 3 days; 0 new payees. receiver: 0 in, 0 out last day; 0 in, 0 out last 3 days; 0 new payers. pair seen 0 times. sender Partnership with 7 accounts, receiver Partnership with 7 accounts. same owner yes, same bank yes.

laundering:
amount 5.6 log10, 2.6 vs typical for currency. hour 0. sender: 1662 out, 1 in last day; 1662 out, 1 in last 3 days; 1001 new payees. receiver: 2 in, 0 out last day; 2 in, 0 out last 3 days; 1 new payers. pair seen 2 times. sender Sole Proprietorship with 6 accounts, receiver Sole Proprietorship with 1 accounts. same owner no, same bank no.



Each transaction becomes one sentence containing all 22 features and nothing else, keeping the comparison with LightGBM on equal information. Hour is converted back from its sine and cosine encoding into a plain number, which a language model reads naturally; no information is added or lost. Numbers are rounded to keep inputs short, since tokenizers split long numbers into many pieces. Missing values are written as "unknown".

The sentences are readable by a person. The normal example has no history because it falls in the first hour of the data, so every count is zero; LightGBM sees the same zeros. The laundering example is a hub sender: 1,662 payments out in one day to 1,001 new payees, with an amount about 400 times the typical size for its currency.

In [3]:
from transformers import AutoTokenizer

MODEL = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

rng = np.random.default_rng(42)
pos = np.flatnonzero(y_train == 1)
neg = rng.choice(np.flatnonzero(y_train == 0), size=200_000, replace=False)
sample = np.sort(np.concatenate([pos, neg]))

train_text = to_text(X_train.iloc[sample]).tolist()
train_labels = y_train[sample]
val_text = to_text(X_val).tolist()

lengths = pd.Series([len(ids) for ids in tokenizer(train_text[:5000])["input_ids"]])
print(f"train sample: {len(train_text):,} rows, {train_labels.mean():.2%} laundering")
print(f"validation: {len(val_text):,} rows")
print(lengths.describe()[["mean", "max"]])

train sample: 202,530 rows, 1.25% laundering
validation: 965,524 rows
mean     99.1038
max     103.0000
dtype: float64


Training uses all 2,530 laundering rows plus a random 200,000 normal rows (seed 42), so 1.25% of the sample is laundering instead of 0.08%. Downsampling normal rows shifts the model's scores but not their order, and PR-AUC and recall at a fixed false positive rate depend only on order. Validation is kept complete, all 965,524 rows at the real rate, so results compare directly with LightGBM.

Sentences are 99 tokens on average and at most 103 in a sample of 5,000, so a maximum length of 128 never cuts a sentence short.

In [5]:
from torch.utils.data import Dataset

MAX_LEN = 128


class TextDataset(Dataset):
    """Holds sentences and labels; tokenizing happens per batch."""

    def __init__(self, texts, labels):
        self.texts, self.labels = texts, labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return {"text": self.texts[i], "label": int(self.labels[i])}


def collate(batch):
    """Tokenize one batch of sentences, padded to the longest in the batch."""
    enc = tokenizer(
        [b["text"] for b in batch],
        truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt",
        return_token_type_ids=False,
    )
    enc["labels"] = torch.tensor([b["label"] for b in batch])
    return enc


train_ds = TextDataset(train_text, train_labels)
batch = collate([train_ds[i] for i in range(4)])
print({k: tuple(v.shape) for k, v in batch.items()})

{'input_ids': (4, 103), 'attention_mask': (4, 103), 'labels': (4,)}


Sentences are tokenized per batch rather than all at once, which keeps memory low: tokenizing all 965,524 validation sentences upfront would hold about 100 million numbers in RAM. Each batch is padded only to its longest sentence, and the attention mask tells the model to ignore the padding. Token type IDs are switched off because DistilBERT does not use them.

In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params / 1e6:.1f}M")
print(model.pre_classifier)
print(model.classifier)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1385.57it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


parameters: 67.0M
Linear(in_features=768, out_features=768, bias=True)
Linear(in_features=768, out_features=2, bias=True)


The model is DistilBERT, a 6 layer distilled version of BERT, with a two class classification head. It fits in the GPU memory available and is a standard, well understood baseline for text classification.

The pretrained body loaded fully. The load report shows the pretraining head (`vocab_*`, used to predict masked words) was dropped and a new classification head (`pre_classifier`, `classifier`) was initialised at random, as expected. The model has 67.0M parameters.

In [7]:
from sklearn.metrics import average_precision_score
from transformers import Trainer, TrainingArguments

# A fixed random fifth of validation, to compare checkpoints during training.
eval_idx = np.random.default_rng(0).choice(len(val_text), size=200_000, replace=False)
eval_ds = TextDataset([val_text[i] for i in eval_idx], y_val[eval_idx])


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    scores = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {"pr_auc": average_precision_score(labels, scores)}


args = TrainingArguments(
    output_dir="../models/distilbert",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    bf16=True,
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="pr_auc",
    greater_is_better=True,
    logging_steps=200,
    remove_unused_columns=False,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collate,
    compute_metrics=compute_metrics,
)

steps = len(train_ds) // args.per_device_train_batch_size * args.num_train_epochs
print(f"eval rows: {len(eval_ds):,} with {int(y_val[eval_idx].sum())} laundering")
print(f"training steps: about {steps:,}")

eval rows: 200,000 with 216 laundering
training steps: about 12,658


Training runs for 2 epochs at batch size 32, about 12,658 steps, with the learning rate warmed up over 500 steps and then decayed. These are standard settings for fine-tuning BERT models; no hyperparameter search is run, matching the fixed settings used for LightGBM. Checkpoints are compared every 2,000 steps on a fixed random fifth of validation (200,000 rows, 216 laundering), and the best one is kept. The final score uses all of validation.

In [8]:
import time

start = time.time()
result = trainer.train()
print(f"training time: {(time.time() - start) / 60:.1f} min")

Step,Training Loss,Validation Loss,Pr Auc
2000,0.055228,0.006655,0.058301
4000,0.048476,0.008326,0.092312
6000,0.051188,0.008082,0.106825
8000,0.042591,0.007076,0.117877
10000,0.036884,0.007016,0.214623
12000,0.043857,0.006945,0.215604
12660,0.043755,0.006892,0.215714


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.59it/s]


training time: 46.1 min


Training took 46 minutes on the laptop GPU (RTX 3060, 6 GB), at about 6 steps per second. PR-AUC on the evaluation fifth rose slowly through the first epoch (0.058 to 0.118), jumped to 0.215 early in the second epoch, then levelled off at 0.216. Validation loss changed little while PR-AUC rose, because training on 1.25% laundering inflates scores at the real 0.1% rate; checkpoints are therefore selected on PR-AUC, not loss. The final checkpoint was the best and is the one kept.

In [9]:
from bankguard.evaluate import evaluate

start = time.time()
pred = trainer.predict(TextDataset(val_text, y_val))
val_scores = torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()
np.save("../models/distilbert/val_scores.npy", val_scores)
print(f"scoring time: {(time.time() - start) / 60:.1f} min")

pd.DataFrame([
    {"model": "LightGBM, owner attributes", "pr_auc": 0.3412,
     "recall_at_0.1%_fpr": 0.3793, "recall_at_1.0%_fpr": 0.7635},
    evaluate(y_val, val_scores, "DistilBERT, same features as text"),
]).round(4)

scoring time: 14.9 min


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,"LightGBM, owner attributes",0.3412,0.3793,0.7635
1,"DistilBERT, same features as text",0.2281,0.2963,0.7259


On all of validation, DistilBERT reaches PR-AUC 0.228 against 0.341 for LightGBM on the same 22 features, and 29.6% against 37.9% recall at a 0.1% false positive rate. At a 1% false positive rate the gap is much smaller: 72.6% against 76.4%. The transformer finds most of the same laundering at a moderate alert budget but ranks less sharply at the top, where the most confident alerts sit. A likely cause is that a text model reads numbers as token pieces and compares them only approximately, while gradient boosting splits on exact values. Scoring all 965,524 validation rows took 15 minutes.

In [10]:
import mlflow
import mlflow.lightgbm

from bankguard.model import predict

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
runs = mlflow.search_runs(experiment_names=["bankguard-laundering"], order_by=["start_time DESC"])
run_id = runs.loc[0, "run_id"]

lgbm = mlflow.lightgbm.load_model(f"runs:/{run_id}/model")
pre = mlflow.artifacts.load_dict(f"runs:/{run_id}/preprocessing.json")
lgbm_scores = predict(lgbm, X_val, pre["features"], pre["categories"])

print("run:", run_id, "| trees:", lgbm.best_iteration_)
print(evaluate(y_val, lgbm_scores, "LightGBM, loaded from MLflow"))

run: 10ac031249824f1b91777d9b4fe463f1 | trees: 1156
{'model': 'LightGBM, loaded from MLflow', 'pr_auc': 0.3412270398403899, 'recall_at_0.1%_fpr': 0.37934362934362936, 'recall_at_1.0%_fpr': 0.7635135135135135}


The LightGBM model is loaded from its MLflow run (10ac0312, 1,156 trees) together with its saved preprocessing, rather than retrained. Scoring validation with it reproduces the training result exactly (PR-AUC 0.3412), which confirms that the saved model and `preprocessing.json` are enough to score new data the same way as in training. The service in step 5 will load the model the same way.

In [12]:
import importlib

import bankguard.data
import bankguard.evaluate

importlib.reload(bankguard.data)
importlib.reload(bankguard.evaluate)

from bankguard.data import load_patterns, typology_for
from bankguard.evaluate import laundering_group, recall_by_group

patterns = load_patterns(DATA / "HI-Small_Patterns.txt")
val_group = laundering_group(typology_for(val, patterns), X_val["sender_out_3d"])

pd.concat(
    {
        "LightGBM": recall_by_group(y_val, lgbm_scores, val_group),
        "DistilBERT": recall_by_group(y_val, val_scores, val_group),
    },
    axis=1,
).round(3)

LightGBM                                        \
                          laundering recall_at_1.0%_fpr recall_at_0.1%_fpr   
patterned                        657              0.913              0.531   
unpatterned, quiet sender        246              0.764              0.179   
unpatterned, hub sender          133              0.023              0.000   

                          DistilBERT                                        
                          laundering recall_at_1.0%_fpr recall_at_0.1%_fpr  
patterned                        657              0.854              0.399  
unpatterned, quiet sender        246              0.772              0.183  
unpatterned, hub sender          133              0.008              0.000

At a 1% false positive rate, LightGBM's lead over DistilBERT comes entirely from patterned laundering (91.3% against 85.4% recall). On unpatterned laundering from quiet senders the two models are level (76.4% against 77.2%, a difference of about 2 of 246 cases). Laundering from hub senders is missed by both (2.3% and 0.8%). That two very different learners fail on the same group, given the same features, supports the conclusion from `02_baselines.ipynb` that these features carry no signal for hub sent laundering, rather than the model being at fault. The LightGBM figures match those in `02_baselines.ipynb`, which confirms the packaged group analysis reproduces it.

In [13]:
def caught_at(scores, fpr=0.01):
    """Which rows score above the line that flags `fpr` of normal transactions."""
    threshold = np.quantile(scores[y_val == 0], 1 - fpr)
    return scores > threshold


laund = y_val == 1
lg, db = caught_at(lgbm_scores)[laund], caught_at(val_scores)[laund]
overlap = pd.DataFrame({
    "group": val_group[laund],
    "both": lg & db,
    "only LightGBM": lg & ~db,
    "only DistilBERT": ~lg & db,
    "neither": ~lg & ~db,
})
display(overlap.groupby("group").sum())

# Combine by averaging each model's rank, so the two score scales do not matter.
combined = (
    pd.Series(lgbm_scores).rank(pct=True) + pd.Series(val_scores).rank(pct=True)
).to_numpy() / 2

pd.DataFrame([
    evaluate(y_val, lgbm_scores, "LightGBM"),
    evaluate(y_val, val_scores, "DistilBERT"),
    evaluate(y_val, combined, "Average of ranks"),
]).round(4)

,both,only LightGBM,only DistilBERT,neither
group,,,,
patterned,541,59,20,37
"unpatterned, hub sender",0,3,1,129
"unpatterned, quiet sender",178,10,12,46


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,LightGBM,0.3412,0.3793,0.7635
1,DistilBERT,0.2281,0.2963,0.7259
2,Average of ranks,0.2910,0.3533,0.7500


The two models largely catch the same laundering: at a 1% false positive rate each, 719 cases are caught by both, 72 only by LightGBM and 33 only by DistilBERT. On quiet senders they nearly agree case by case (178 together, 10 and 12 apart), and the 46 cases caught by neither suggest a ceiling of about 81% recall on that group with these features. Averaging the two models' ranks, with equal weights fixed in advance, scores PR-AUC 0.291, below LightGBM alone: DistilBERT's few extra catches come with extra false alarms that push good LightGBM alerts down the ranking. DistilBERT therefore adds nothing alongside LightGBM. Weights tuned on validation were not tried, since choosing them on the same data would make the result optimistic.

In [14]:
trainer.save_model("../models/distilbert/best")
tokenizer.save_pretrained("../models/distilbert/best")

metrics = evaluate(y_val, val_scores, "DistilBERT, same features as text")

mlflow.set_experiment("bankguard-laundering")
with mlflow.start_run(run_name=metrics["model"]):
    mlflow.log_params({
        "model_name": MODEL,
        "epochs": args.num_train_epochs,
        "batch_size": args.per_device_train_batch_size,
        "learning_rate": args.learning_rate,
        "warmup_steps": args.warmup_steps,
        "weight_decay": args.weight_decay,
        "max_len": MAX_LEN,
        "train_rows": len(train_ds),
        "train_laundering_share": round(float(train_labels.mean()), 4),
        "seed": args.seed,
        "n_features": len(FEATURES),
    })
    mlflow.log_metrics({k.replace("%", "pct"): v for k, v in metrics.items() if k != "model"})
    mlflow.log_metric("train_minutes", result.metrics["train_runtime"] / 60)
    mlflow.set_tag("best_checkpoint", trainer.state.best_model_checkpoint)
    mlflow.set_tag("model_dir", "models/distilbert/best")

mlflow.search_runs(experiment_names=["bankguard-laundering"])[
    ["tags.mlflow.runName", "metrics.pr_auc", "metrics.recall_at_0.1pct_fpr", "metrics.recall_at_1.0pct_fpr"]
].round(4)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.05s/it]


,tags.mlflow.runName,metrics.pr_auc,metrics.recall_at_0.1pct_fpr,metrics.recall_at_1.0pct_fpr
0,"DistilBERT, same features as text",0.2281,0.2963,0.7259
1,"LightGBM, owner attributes",0.3412,0.3793,0.7635


The DistilBERT run is logged to the same MLflow experiment as LightGBM, with its training settings, sampling choices, full validation metrics and training time. The best checkpoint and its tokenizer are saved together in `models/distilbert/best`. The model file is not copied into MLflow because it is not the model that will be served; the logged settings and the saved folder are enough to reproduce or reload it.

## ModernBERT

The same pipeline with a stronger encoder. ModernBERT (2024) is a redesigned BERT with about 150M parameters, trained on far more text than DistilBERT, including code. The question: does a larger, more recent encoder narrow the gap to LightGBM on the same sentences?

In [15]:
distil_scores = val_scores  # keep DistilBERT's scores under their own name
del trainer, model
torch.cuda.empty_cache()

MODEL = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

lengths = pd.Series([len(ids) for ids in tokenizer(train_text[:5000])["input_ids"]])
print(lengths.describe()[["mean", "max"]])

example = "sender: 1662 out, 1 in last day; 1001 new payees."
distil_tok = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
print("DistilBERT:", distil_tok.tokenize(example))
print("ModernBERT:", tokenizer.tokenize(example))

mean     99.2008
max     107.0000
dtype: float64
DistilBERT: ['send', '##er', ':', '1662', 'out', ',', '1', 'in', 'last', 'day', ';', '100', '##1', 'new', 'pay', '##ees', '.']
ModernBERT: ['sender', ':', 'Ġ16', '62', 'Ġout', ',', 'Ġ1', 'Ġin', 'Ġlast', 'Ġday', ';', 'Ġ100', '1', 'Ġnew', 'Ġpay', 'ees', '.']


ModernBERT's tokenizer gives sentences of about the same length (mean 99 tokens, at most 107), so the maximum length of 128 still applies. Comparing the two tokenizers on the same sentence shows how text models see numbers: DistilBERT keeps 1662 as one token but splits 1001 into 100 and 1, while ModernBERT splits 1662 into 16 and 62. In both cases the pieces follow the vocabulary rather than the value, so the model must learn from examples that one sequence of pieces is a larger number than another. Gradient boosting compares the values directly.

In [22]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

probe_args = TrainingArguments(
    output_dir="../models/modernbert-probe",
    max_steps=100,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    bf16=True,
    logging_steps=50,
    save_strategy="no",
    eval_strategy="no",
    remove_unused_columns=False,
    report_to="none",
    seed=42,
)

torch.cuda.reset_peak_memory_stats()
probe = Trainer(model=model, args=probe_args, train_dataset=train_ds, data_collator=collate)
out = probe.train()
print(f"steps per second: {out.metrics['train_steps_per_second']:.2f}")
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 479.93it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


parameters: 149.6M


Step,Training Loss
50,0.121677
100,0.068188


steps per second: 3.26
peak GPU memory: 4.30 GB


A 100 step probe on the laptop ran at 1.12 steps per second with 4.3 GB of peak GPU memory, against 6.05 steps per second for DistilBERT. The laptop was short of both GPU memory (about 0.5 GB free) and RAM (96% in use), and ModernBERT's faster attention path is not available on Windows. A full run was estimated at about 6 hours, so training moved to Kaggle (`03b_modernbert_kaggle.ipynb`).

In [23]:
import hashlib

print("train text:", hashlib.md5("\n".join(train_text).encode()).hexdigest())
print("val text:  ", hashlib.md5("\n".join(val_text).encode()).hexdigest())

train text: 6b7502e690497d6d06dbd6ae12a15e96
val text:   ed4defe73ab6fe9ee28e95ea9e7f12b3


ModernBERT was trained on Kaggle (two T4 GPUs) because the laptop ran short of GPU memory and RAM. The MD5 hashes of the training and validation sentences on the laptop match those computed on Kaggle (train 6b7502e6, validation ed4defe7), so both models read exactly the same inputs. Kaggle used fp16 instead of bf16, which the T4 does not support, and 16 sentences per GPU across two GPUs, keeping 32 sentences per step as for DistilBERT.

In [24]:
import json

mb_dir = Path("../models/modernbert")
modernbert_scores = np.load(mb_dir / "modernbert_val_scores.npy")
mb_metrics = json.loads((mb_dir / "modernbert_metrics.json").read_text())

combined_mb = (
    pd.Series(lgbm_scores).rank(pct=True) + pd.Series(modernbert_scores).rank(pct=True)
).to_numpy() / 2

display(pd.DataFrame([
    evaluate(y_val, lgbm_scores, "LightGBM"),
    evaluate(y_val, distil_scores, "DistilBERT"),
    evaluate(y_val, modernbert_scores, "ModernBERT"),
    evaluate(y_val, combined_mb, "LightGBM + ModernBERT, average of ranks"),
]).round(4))

display(pd.concat({
    "LightGBM": recall_by_group(y_val, lgbm_scores, val_group),
    "ModernBERT": recall_by_group(y_val, modernbert_scores, val_group),
}, axis=1).round(3))

lg, mb = caught_at(lgbm_scores)[laund], caught_at(modernbert_scores)[laund]
pd.DataFrame({
    "group": val_group[laund],
    "both": lg & mb,
    "only LightGBM": lg & ~mb,
    "only ModernBERT": ~lg & mb,
    "neither": ~lg & ~mb,
}).groupby("group").sum()

,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,LightGBM,0.3412,0.3793,0.7635
1,DistilBERT,0.2281,0.2963,0.7259
2,ModernBERT,0.3023,0.3600,0.7828
3,"LightGBM + ModernBERT, average of ranks",0.3361,0.3774,0.7712


LightGBM                                        \
                          laundering recall_at_1.0%_fpr recall_at_0.1%_fpr   
patterned                        657              0.913              0.531   
unpatterned, quiet sender        246              0.764              0.179   
unpatterned, hub sender          133              0.023              0.000   

                          ModernBERT                                        
                          laundering recall_at_1.0%_fpr recall_at_0.1%_fpr  
patterned                        657              0.930              0.489  
unpatterned, quiet sender        246              0.813              0.211  
unpatterned, hub sender          133              0.000              0.000

,both,only LightGBM,only ModernBERT,neither
group,,,,
patterned,585,15,26,31
"unpatterned, hub sender",0,3,0,130
"unpatterned, quiet sender",184,4,16,42


ModernBERT, trained on Kaggle with the same settings and sentences, reaches PR-AUC 0.302 against 0.228 for DistilBERT and 0.341 for LightGBM. The stronger encoder closes most of the gap, and at a 1% false positive rate it catches more laundering than LightGBM (78.3% against 76.4%). Its largest gain is on unpatterned laundering from quiet senders, 81.3% against 76.4%; at a 1% budget each, it catches 16 quiet sender cases that LightGBM misses and LightGBM catches 4 that it misses. LightGBM remains sharper at the very top of the ranking, catching more patterned laundering at a 0.1% budget (53.1% against 48.9%), which gives it the higher PR-AUC. An equal-weight average of ranks lands between the two models on every metric. Neither model detects laundering from hub senders. ModernBERT's validation PR-AUC was still rising at the last checkpoint (0.290 on the evaluation fifth), so a longer run may improve it further; these figures come from a single seed.

In [25]:
mlflow.set_experiment("bankguard-laundering")
with mlflow.start_run(run_name=mb_metrics["model"]):
    mlflow.log_params({
        "model_name": "answerdotai/ModernBERT-base",
        "epochs": 2,
        "batch_size": 32,
        "per_gpu_batch_size": 16,
        "gpus": 2,
        "learning_rate": 2e-5,
        "warmup_steps": 500,
        "weight_decay": 0.01,
        "precision": "fp16",
        "max_len": MAX_LEN,
        "train_rows": len(train_text),
        "train_laundering_share": round(float(train_labels.mean()), 4),
        "seed": 42,
        "n_features": len(FEATURES),
    })
    for key in ["pr_auc", "recall_at_0.1%_fpr", "recall_at_1.0%_fpr"]:
        mlflow.log_metric(key.replace("%", "pct"), mb_metrics[key])
    mlflow.log_metric("train_minutes", mb_metrics["train_minutes"])

    # The training curve, one point per logged step, so MLflow can chart it.
    for entry in mb_metrics["log_history"]:
        if "eval_pr_auc" in entry:
            mlflow.log_metric("eval_pr_auc", entry["eval_pr_auc"], step=entry["step"])
        if "loss" in entry:
            mlflow.log_metric("train_loss", entry["loss"], step=entry["step"])

    mlflow.log_artifact(str(mb_dir / "modernbert_metrics.json"))
    mlflow.set_tags({
        "trained_on": "Kaggle, 2x Tesla T4",
        "notebook": "notebooks/03b_modernbert_kaggle.ipynb",
        "best_checkpoint": mb_metrics["best_checkpoint"],
    })

mlflow.search_runs(experiment_names=["bankguard-laundering"])[
    ["tags.mlflow.runName", "metrics.pr_auc", "metrics.recall_at_0.1pct_fpr", "metrics.recall_at_1.0pct_fpr"]
].round(4)

,tags.mlflow.runName,metrics.pr_auc,metrics.recall_at_0.1pct_fpr,metrics.recall_at_1.0pct_fpr
0,"ModernBERT, same features as text",0.3023,0.3600,0.7828
1,"DistilBERT, same features as text",0.2281,0.2963,0.7259
2,"LightGBM, owner attributes",0.3412,0.3793,0.7635


The ModernBERT run is logged to MLflow from the metrics file saved on Kaggle, with its settings, where it ran, and its training curve logged step by step so MLflow can chart it. This two epoch run is kept as a result in its own right: its validation PR-AUC was still rising at the last checkpoint, so a four epoch run with evaluation every 4,000 steps was started on Kaggle to test whether it improves further.

## ModernBERT, four epochs

Because validation PR-AUC was still rising after two epochs, ModernBERT was trained again for four epochs, evaluating every 4,000 steps (`03b_modernbert_kaggle.ipynb`, second version). Validation PR-AUC rose from 0.302 to 0.353, above LightGBM's 0.341, while recall at a 0.1% false positive rate (0.375 against 0.379) and at 1% (0.761 against 0.764) stayed just below LightGBM. The training curve flattened at the end (0.330 to 0.332 on the evaluation fifth over the last checkpoints), so four epochs is close to convergence. This is a single run, and the gap to LightGBM is about the size of LightGBM's own spread across seeds (0.331 to 0.342), so the two models are best described as level. Both runs are kept in the comparison; selecting the better of two runs on validation makes its validation score slightly optimistic, which the final test set score accounts for.

In [1]:
import json
from pathlib import Path

import mlflow

mb4 = json.loads(Path("../models/modernbert/modernbert_4ep_metrics.json").read_text())

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("bankguard-laundering")
with mlflow.start_run(run_name=mb4["model"]):
    mlflow.log_params({
        "model_name": "answerdotai/ModernBERT-base", "epochs": 4, "eval_steps": 4000,
        "batch_size": 32, "per_gpu_batch_size": 16, "gpus": 2, "learning_rate": 2e-5,
        "warmup_steps": 500, "weight_decay": 0.01, "precision": "fp16", "max_len": 128,
        "train_rows": 202_530, "train_laundering_share": 0.0125, "seed": 42, "n_features": 22,
    })
    for key in ["pr_auc", "recall_at_0.1%_fpr", "recall_at_1.0%_fpr"]:
        mlflow.log_metric(key.replace("%", "pct"), mb4[key])
    mlflow.log_metric("train_minutes", mb4["train_minutes"])
    for entry in mb4["log_history"]:
        if "eval_pr_auc" in entry:
            mlflow.log_metric("eval_pr_auc", entry["eval_pr_auc"], step=entry["step"])
        if "loss" in entry:
            mlflow.log_metric("train_loss", entry["loss"], step=entry["step"])
    mlflow.log_artifact("../models/modernbert/modernbert_4ep_metrics.json")
    mlflow.set_tags({
        "trained_on": "Kaggle, 2x Tesla T4",
        "notebook": "notebooks/03b_modernbert_kaggle.ipynb (version 2)",
        "best_checkpoint": mb4["best_checkpoint"],
    })

mlflow.search_runs(experiment_names=["bankguard-laundering"])[
    ["tags.mlflow.runName", "metrics.pr_auc", "metrics.recall_at_0.1pct_fpr", "metrics.recall_at_1.0pct_fpr"]
].round(4)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,tags.mlflow.runName,metrics.pr_auc,metrics.recall_at_0.1pct_fpr,metrics.recall_at_1.0pct_fpr
0,"ModernBERT, 4 epochs",0.3526,0.3755,0.7606
1,"ModernBERT, same features as text",0.3023,0.3600,0.7828
2,"DistilBERT, same features as text",0.2281,0.2963,0.7259
3,"LightGBM, owner attributes",0.3412,0.3793,0.7635


## Test set predictions

The saved DistilBERT model scores the test rows (9 to 10 Sep) for the reference table in the comparison notebook. The model choice is already locked, so this score is for reporting only. A check on validation confirms the reloaded model matches the original run.

In [7]:
import hashlib
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

from bankguard.data import load_accounts, load_transactions, split_by_date
from bankguard.features import FEATURES, build_features, fit_currency_median
from bankguard.text import to_text

DATA = Path("../data")
tx = load_transactions(DATA / "HI-Small_Trans.csv")
accounts = load_accounts(DATA / "HI-Small_accounts.csv")
splits = split_by_date(tx)
X = build_features(tx, accounts, fit_currency_median(splits["train"]))
val_text = to_text(X.loc[splits["validation"].index, FEATURES]).tolist()
test_text = to_text(X.loc[splits["test"].index, FEATURES]).tolist()
del tx, X
print("val text:", hashlib.md5("\n".join(val_text).encode()).hexdigest())

BEST = "../models/distilbert/best"
tokenizer = AutoTokenizer.from_pretrained(BEST)
model = AutoModelForSequenceClassification.from_pretrained(BEST)


class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return {"text": self.texts[i]}


def collate(batch):
    return tokenizer([b["text"] for b in batch], truncation=True, max_length=128, padding=True,
                     return_tensors="pt", return_token_type_ids=False)


scorer = Trainer(
    model=model,
    args=TrainingArguments(output_dir="../models/distilbert/predict", per_device_eval_batch_size=128,
                           bf16=True, remove_unused_columns=False, report_to="none"),
    data_collator=collate,
)


def score(texts):
    logits = scorer.predict(TextDataset(texts)).predictions
    return torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()


saved = np.load("../models/distilbert/val_scores.npy")[:5_000]
print(f"largest difference from saved validation scores: {np.abs(score(val_text[:5_000]) - saved).max():.2e}")

test_scores = score(test_text)
np.save("../models/distilbert/test_scores.npy", test_scores)
print(f"test rows: {len(test_text):,}; missing: {int(np.isnan(test_scores).sum())}")

val text: ed4defe73ab6fe9ee28e95ea9e7f12b3


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1179.73it/s]


largest difference from saved validation scores: 4.14e-04


test rows: 862,792; missing: 0


The validation sentences give the same hash as at training time (ed4defe7), and the reloaded model reproduces its saved validation scores to within 4.1e-4, the small rounding expected from bf16 arithmetic. All 862,792 test rows are scored and saved without reading the test labels.